In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=== RANDOM FOREST v4 - COLDTRACK (Versión Final Ajustada) ===\n")

# Cargar datos
X = pd.read_csv("X_features.csv")
y = pd.read_csv("y_target.csv").squeeze()

# Limpieza de features (manteniendo solo las más predictivas y tempranas)
features_to_remove = [
    'high_temp', 'critical_vibration', 'compressor_overwork', 'long_door_open',
    'ConsumoElectrico', 'Vibration', 
    'inTemp_std_5min', 'Vibration_std_5min', 'inTemp_std_15min'
]

X_clean = X.drop(columns=features_to_remove, errors='ignore')

print(f"Features finales: {X_clean.shape[1]} columnas")

# Modelo con parámetros más conservadores
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# División temporal
tscv = TimeSeriesSplit(n_splits=5)
train_index, test_index = list(tscv.split(X_clean))[-1]

X_train = X_clean.iloc[train_index]
X_test  = X_clean.iloc[test_index]
y_train = y.iloc[train_index]
y_test  = y.iloc[test_index]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}\n")

model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# ====================== RESULTADOS FINALES ======================
print("="*70)
print("RESULTADOS FINALES - MODELO v4")
print("="*70)
print(f"Accuracy     : {y_pred.mean():.4f}")
print(f"ROC AUC      : {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)']))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

# Importancia de features
importancia = pd.DataFrame({
    'Feature': X_clean.columns,
    'Importancia': model.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\nTop 10 Features más importantes:")
print(importancia.head(10))

# Guardar modelo final
joblib.dump(model, "coldtrack_rf_model_final.pkl")
joblib.dump(X_clean.columns.tolist(), "feature_columns_final.pkl")

print("\n✅ Modelo FINAL guardado como: coldtrack_rf_model_final.pkl")
print("✅ Columnas guardadas como: feature_columns_final.pkl")

=== RANDOM FOREST v4 - COLDTRACK (Versión Final Ajustada) ===

Features finales: 16 columnas
Train: 259,200 | Test: 51,840

RESULTADOS FINALES - MODELO v4
Accuracy     : 0.0858
ROC AUC      : 0.9994

Reporte de Clasificación:
                   precision    recall  f1-score   support

       Normal (0)       1.00      0.99      1.00     47722
Mantenimiento (1)       0.93      1.00      0.96      4118

         accuracy                           0.99     51840
        macro avg       0.96      1.00      0.98     51840
     weighted avg       0.99      0.99      0.99     51840


Matriz de Confusión:
[[47390   332]
 [    0  4118]]

Top 10 Features más importantes:
               Feature  Importancia
1              InHumid     0.396168
4   temp_diff_setpoint     0.253513
0               inTemp     0.214478
2              outTemp     0.087795
3             outHumid     0.035401
6      inTemp_ma_15min     0.002935
5       inTemp_ma_5min     0.002913
8    Vibration_ma_5min     0.002581
7     